<a href="https://colab.research.google.com/github/UltraGigaHero/EU_M_Math/blob/main/Chap08_Ex_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import requests, zipfile
import io

from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression

url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data'
res = requests.get(url).content

auto = pd.read_csv(io.StringIO(res.decode('utf-8')), header = None)

auto.columns = ['symboling', 'normalized-losses', 'make', 'fuel-type', 'aspiration', 'num-of-doors',
               'body-style', 'drive-wheels', 'engine-location', 'wheel-base', 'lenght', 'width', 'height',
               'curb-weight', 'engine-type', 'num-of-cylinders', 'engine-size', 'fuel-system', 'bore',
               'stroke', 'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 'price']

auto = auto[['price', 'width', 'engine-size']]
auto.isin(['?']).sum()

auto = auto.replace('?', np.nan).dropna()

auto = auto.assign(price = pd.to_numeric(auto.price))
auto = auto.assign(width = pd.to_numeric(auto.width))
auto = auto.assign(**{'engine-size': pd.to_numeric(auto['engine-size'])})

X = auto.drop('price', axis=1)
y = auto['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=0)

linear = LinearRegression()
Lasso = Lasso(alpha=200)

for model in [linear, Lasso]:
    model.fit(X_train, y_train)
    print('{}(train):{:.6f}'.format(model.__class__.__name__, model.score(X_train, y_train)))
    print('{}(test):{:.6f}'.format(model.__class__.__name__, model.score(X_test, y_test)))

LinearRegression(train):0.783189
LinearRegression(test):0.778292
Lasso(train):0.782839
Lasso(test):0.782421


# ラッソ回帰とリッジ回帰における正則化項の目的とメリット

## 1. 正則化項とは

通常の線形回帰では、損失関数（誤差）を最小化することでパラメータ **w** を推定する。

$$
E = \|\boldsymbol{y} - \boldsymbol{\Phi}\boldsymbol{w}\|^2
$$

しかし、説明変数間に強い相関がある場合（多重共線性）や、パラメータ数がサンプル数に比べて多い場合、
**w** の絶対値が非常に大きくなり、過学習が発生する。

正則化とは、この損失関数に **パラメータの大きさへのペナルティ項** を加える手法であり、
リッジ回帰（L2正則化）とラッソ回帰（L1正則化）はペナルティの種類が異なる。

---

## 2. リッジ回帰（Ridge / L2正則化）

### 損失関数
$$
E_{\text{Ridge}} = \|\boldsymbol{y} - \boldsymbol{\Phi}\boldsymbol{w}\|^2 + \alpha\|\boldsymbol{w}\|_2^2
$$

正則化項は **L2ノルム**（係数の二乗和）：
$$
\|\boldsymbol{w}\|_2^2 = w_0^2 + w_1^2 + \cdots + w_d^2
$$

### 目的とメリット

| 観点 | 内容 |
|------|------|
| **多重共線性** | 説明変数間の相関が高い場合、$\boldsymbol{\Phi}^T\boldsymbol{\Phi}$ が正則でなくなり逆行列が不安定になる。リッジ回帰は対角要素に $\alpha$ を加えた $(\boldsymbol{\Phi}^T\boldsymbol{\Phi} + \alpha\boldsymbol{I})^{-1}$ を用いることで行列を必ず正則にし、安定した係数推定を実現する。 |
| **過学習** | 係数が大きくなりすぎることにペナルティを課し、モデルがノイズに過剰適合するのを防ぐ。訓練データへの適合度とモデルの複雑さのバランスをとる。 |
| **パラメータ選択** | 係数はゼロには**ならず**、全ての特徴量をモデルに残したまま係数を小さく縮小する。ハイパーパラメータ $\alpha$ を大きくするほど係数の縮小が強くなる。 |

---

## 3. ラッソ回帰（Lasso / L1正則化）

### 損失関数
$$
E_{\text{Lasso}} = \|\boldsymbol{y} - \boldsymbol{\Phi}\boldsymbol{w}\|^2 + \alpha\|\boldsymbol{w}\|_1
$$

正則化項は **L1ノルム**（係数の絶対値の和）：
$$
\|\boldsymbol{w}\|_1 = |w_0| + |w_1| + \cdots + |w_d|
$$

### 目的とメリット

| 観点 | 内容 |
|------|------|
| **多重共線性** | 相関の高い特徴量のうち、重要でない方の係数を**完全にゼロ**にすることで自動的に変数を選択し、多重共線性の影響を排除する。 |
| **過学習** | 不要なパラメータをゼロにしてモデルをシンプルにすることで、過学習を防ぐ。 |
| **パラメータ選択** | 係数を**スパース（疎）**にする（多くの係数がちょうど0になる）という最大の特徴を持つ。特徴量が多く、重要な変数を自動的に絞り込みたい場合に有効。モデルの解釈性が高くなる。 |

---

## 4. 比較まとめ

| 比較項目 | リッジ回帰（L2） | ラッソ回帰（L1） |
|----------|----------------|----------------|
| 正則化項 | 係数の二乗和 | 係数の絶対値の和 |
| 係数のゼロ化 | ゼロに近づけるが**完全にはゼロにならない** | 不要な係数が**完全にゼロ**になる |
| 多重共線性への対応 | 全変数を保持したまま安定化 | 相関の高い変数の一方を除去 |
| 過学習の防止 | 係数を縮小して汎化性能向上 | 変数を削減してモデルを単純化 |
| パラメータ選択 | 暗黙的（全変数を使用） | 明示的（自動的な変数選択） |
| ハイパーパラメータ | $\alpha$（交差検証で最適化） | $\alpha$（交差検証で最適化） |
| 向いているケース | 全変数が重要・多重共線性が強い | 不要な変数が多い・解釈性重視 |

---

## 5. ハイパーパラメータ $\alpha$ の選択

$\alpha$ はモデルの複雑さと汎化能力のバランスを制御するハイパーパラメータである。

- $\alpha = 0$：通常の線形回帰と等価（正則化なし）
- $\alpha$ が大きい：ペナルティが強く、係数が強く縮小される（アンダーフィットのリスク）
- $\alpha$ が小さい：ペナルティが弱く、過学習のリスクが残る

最適な $\alpha$ は **交差検証（Cross-Validation）** によって決定する。
scikit-learn では `RidgeCV` や `LassoCV` を用いて自動的に最適な $\alpha$ を探索できる。